# Repo → one markdown file → video → repo

`repo_md` covers the two ends of that pipeline:

- `to_markdown(root, out, include=...)` — every matching file, in order, in one document
- `from_markdown(md, out_dir)` — parse it back into a tree, verifying each file

The middle (film it, transcribe it) happens outside this notebook. What matters is
that each block carries a SHA-256, so transcription errors come back as a **named
list of files to redo** instead of silent corruption. The last section demonstrates
exactly that.

Standard library only — nothing to install on the offline machine.

In [1]:
import pathlib, repo_md

ROOT = pathlib.Path("../..").resolve()
CODE = [".py", ".sh", ".zsh", ".bash", ".toml", ".yaml", ".yml", ".md", ".txt"]

files = repo_md.collect(ROOT, include=CODE)
print(f"{len(files)} files would be written, in this order:\n")
for f in files[:6]:
    print("  ", f)
print("   ...")
for f in files[-3:]:
    print("  ", f)

240 files would be written, in this order:

   .pre-commit-config.yaml
   README.md
   airgap-qr-transfer/README.md
   airgap-qr-transfer/agmini.py
   airgap-qr-transfer/agpack.py
   airgap-qr-transfer/agtype.py
   ...
   zsh_scripts/prevent_main_push.zsh
   zsh_scripts/save_and_src_zshrc.zsh
   zsh_scripts/zsh_backup.zsh


## 1. Write the document

Written outside the repo so a ~600 KB generated file doesn't land in your git status.
Credentials (`.env`, `vault/*`, `*token*`, keys) are dropped before `include` is
applied, so no include list can pull one onto the screen you're about to film.

In [2]:
MD = ROOT.parent / "kretsinger-repo.md"
info = repo_md.to_markdown(ROOT, MD, include=CODE)
info

240 files, 594,531 bytes -> 673,628 characters of markdown
~169 screenfuls at ~4,000 chars each
wrote /Users/Akseldkw/coding/kretsinger-repo.md


{'path': '/Users/Akseldkw/coding/kretsinger-repo.md',
 'files': 240,
 'source_bytes': 594531,
 'source_lines': 16884,
 'markdown_chars': 673628,
 'screens_at_4000_chars': 169}

## 2. What it looks like

A manifest up front, then one delimited block per file.

In [ ]:
doc = MD.read_text()
print(doc[:700])
print("   ...\n")

# the start of one file block
i = doc.find("<!--F")
print(doc[i : i + 520])

# Repository export

Source: `/Users/Akseldkw/coding/kretsinger`

Block delimiter nonce: `49e0d1`

240 files, 594,531 bytes, 16,884 lines.

Rebuild with `repo_md.from_markdown(path, out_dir)`. Each block is delimited by `<!--F49e0d1:path-->` and `<!--E49e0d1:path-->` and carries a SHA-256, so any file that did not survive transcription is reported by name rather than failing silently.

## Manifest

| path | lines | bytes | sha256 |
|---|---:|---:|---|
| `.pre-commit-config.yaml` | 89 | 2646 | `730b7a98cedaf215` |
| `README.md` | 12 | 532 | `102ea587d92f8aa4` |
| `airgap-qr-transfer/README.md` | 318 | 12625 | `7b93e0091fe13095` |
| `airgap-qr-transfer/agmini.py` | 182 | 6764 | `780bf99e9b82e6
   ...

<!--F49e0d1:path-->` and `<!--E49e0d1:path-->` and carries a SHA-256, so any file that did not survive transcription is reported by name rather than failing silently.

## Manifest

| path | lines | bytes | sha256 |
|---|---:|---:|---|
| `.pre-commit-config.yaml` | 89 | 2646 | `730b7a98cedaf

The delimiters carry a per-export nonce (`F3a91c2:` and so on). That is not
decoration: this module contains the delimiter strings as literals, so a fixed
delimiter would split `repo_md.py`'s own block in half when you export the folder
it lives in.

## 3. Rebuild it — the lossless case

Before trusting any of this to a camera, confirm the document rebuilds perfectly
when nothing has corrupted it.

In [ ]:
import shutil, tempfile, hashlib

OUT = pathlib.Path(tempfile.mkdtemp()) / "rebuilt"
res = repo_md.from_markdown(MD, OUT)

mismatched = [
    f
    for f in files
    if hashlib.sha256((OUT / f).read_bytes()).hexdigest() != hashlib.sha256((ROOT / f).read_bytes()).hexdigest()
]
print(f"\nbyte-identical to the source: {not mismatched}")

240 blocks parsed -> 240 verified, 0 corrupt, 0 missing

byte-identical to the source: True


## 4. The case you'll actually hit

Transcription introduces errors. Here we corrupt a handful of characters the way OCR
does — `l`→`1`, `O`→`0`, a dropped space — and confirm the rebuild names precisely the
damaged files.

In [ ]:
import random

rng = random.Random(7)
text = list(doc)
positions = [p for p in rng.sample(range(len(text)), 4000) if text[p] in "lO1 0"][:5]
for p in positions:
    text[p] = {"l": "1", "1": "l", "O": "0", "0": "O", " ": ""}[text[p]]
corrupted = "".join(text)

BAD_MD = pathlib.Path(tempfile.mkdtemp()) / "transcribed.md"
BAD_MD.write_text(corrupted)

OUT2 = pathlib.Path(tempfile.mkdtemp()) / "from-video"
res2 = repo_md.from_markdown(BAD_MD, OUT2)
print(f"\ncorrupted {len(positions)} characters out of {len(doc):,}")
print("files needing another pass:", res2["bad"] or "none")

240 blocks parsed -> 235 verified, 5 corrupt, 0 missing
   re-check airgap-qr-transfer/agpack.py
   re-check airgap-qr-transfer/agtype.py
   re-check airgap-qr-transfer/markdown/repo_md.py
   re-check kret_np_pd/dataset_to_table.py
   re-check kret_sklearn/custom_transformers.py

corrupted 5 characters out of 673,628
files needing another pass: ['airgap-qr-transfer/agpack.py', 'airgap-qr-transfer/agtype.py', 'airgap-qr-transfer/markdown/repo_md.py', 'kret_np_pd/dataset_to_table.py', 'kret_sklearn/custom_transformers.py']


That is the whole reason for the per-file checksums. Re-transcribe only those files,
splice them in, and re-run — you converge instead of hunting for a wrong character in
half a megabyte.

Worth knowing what you're up against: at 99.99% per-character accuracy (strong OCR),
roughly 36 of these 223 files come back damaged, because errors compound across a
~2,300-character file. The loop above is what makes that recoverable.

## 5. Line numbers

`number_lines=True` prefixes every line with `NNNN| `. It costs ~16% more document
to film, and makes a dropped or merged line — the most common transcription failure — visible
at a glance. `from_markdown` detects and strips the prefixes automatically, so the
rebuild is unchanged.

In [ ]:
NUM = ROOT.parent / "kretsinger-repo-numbered.md"
info_n = repo_md.to_markdown(ROOT, NUM, include=CODE, number_lines=True, quiet=True)
print(f"plain    {info['markdown_chars']:>9,} chars   {info['screens_at_4000_chars']:>4} screens")
print(
    f"numbered {info_n['markdown_chars']:>9,} chars   {info_n['screens_at_4000_chars']:>4} screens"
    f"   (+{info_n['markdown_chars']/info['markdown_chars']-1:.0%})"
)

OUT3 = pathlib.Path(tempfile.mkdtemp()) / "from-numbered"
res3 = repo_md.from_markdown(NUM, OUT3, quiet=True)
print(f"\nnumbered document still rebuilds cleanly: " f"{res3['ok']} verified, {len(res3['bad'])} corrupt")

plain      673,628 chars    169 screens
numbered   780,686 chars    196 screens   (+16%)

numbered document still rebuilds cleanly: 240 verified, 0 corrupt


## Filming it

~148 screenfuls at a readable size, so roughly 7–8 minutes at 3 seconds per screen.
Film in sections rather than one continuous take — when a chunk transcribes badly,
you re-shoot that section instead of the whole thing.

The document is also ~148K tokens. Most models won't take that in one pass, so it
gets chunked; chunk boundaries are where lines tend to get dropped or duplicated.
Section 4's check is what catches it.

Clean up the generated documents when you're done:

```python
MD.unlink(); NUM.unlink()
```